In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor

In [12]:
df_pedidos = pd.read_csv('pedidos_historicos_genericos.csv')
df_pronostico = pd.read_csv('pronostico_demanda_generico.csv')
df_capacidad = pd.read_csv('capacidad_logistica_generica.csv')
df_plan = pd.read_csv('plan_entrega_clientes_generico.csv')

df_pedidos['fecha_despacho'] = pd.to_datetime(df_pedidos['fecha_despacho'])
df_pronostico['fecha_despacho'] = pd.to_datetime(df_pronostico['fecha_despacho'])
df_pronostico['fecha_emision'] = pd.to_datetime(df_pronostico['fecha_emision'])
df_plan['fecha_plan'] = pd.to_datetime(df_plan['fecha_plan'])
df_plan['fecha_compra_real'] = pd.to_datetime(df_plan['fecha_compra_real'])

df_reales = df_pedidos.groupby(['centro_distribucion', 'fecha_despacho'], as_index=False)['unidades_reales'].sum()

# Calidad de datos

In [13]:
df_merged_t1 = pd.merge(
    df_pronostico,
    df_reales,
    on=['centro_distribucion', 'fecha_despacho'],
    how='inner'
)

def calcular_metricas(df):
    real = df['unidades_reales'].values
    pred = df['unidades_pronosticadas'].values
    wape = np.sum(np.abs(real - pred)) / np.sum(real)
    bias = np.sum(real - pred) / np.sum(real)
    return pd.Series({'wape': wape, 'bias': bias})

metricas_cd = df_merged_t1.groupby('centro_distribucion', group_keys=False).apply(calcular_metricas, include_groups=False).reset_index()
metricas_horizonte = df_merged_t1.groupby('horizonte', group_keys=False).apply(calcular_metricas, include_groups=False).reset_index()
metricas_cd_horizonte = df_merged_t1.groupby(['centro_distribucion', 'horizonte'], group_keys=False).apply(calcular_metricas, include_groups=False).reset_index()

print("Métricas por Centro de Distribución:")
print(metricas_cd)
print("\nMétricas por Horizonte:")
print(metricas_horizonte)

Métricas por Centro de Distribución:
  centro_distribucion      wape      bias
0           CD_CENTRO  0.232828 -0.202149
1             CD_ESTE  0.258609  0.252978
2            CD_NORTE  0.098169  0.074733
3            CD_OESTE  0.239037 -0.226615
4         CD_PACIFICO  0.221526 -0.215801
5              CD_SUR  0.315566  0.315566

Métricas por Horizonte:
  horizonte      wape      bias
0       D+1  0.197717  0.069390
1       D+2  0.202979  0.075555
2       D+3  0.208471  0.063995
3       D+4  0.225290  0.052406


# Estado Plan de entrega

In [14]:
df_plan = pd.read_csv('plan_entrega_clientes_generico.csv')
df_plan['fecha_plan'] = pd.to_datetime(df_plan['fecha_plan'])
df_plan['fecha_compra_real'] = pd.to_datetime(df_plan['fecha_compra_real'])

total_unidades_plan = df_plan['unidades_plan'].sum()
total_unidades_reales = df_plan['unidades_reales'].sum()

bias_plan_global = (total_unidades_reales - total_unidades_plan) / total_unidades_reales
porcentaje_fuera_plan_total = df_plan['fuera_de_plan'].mean() * 100

fuera_plan_cd = df_plan.groupby('centro_distribucion')['fuera_de_plan'].agg(
    total_pedidos='count',
    pedidos_fuera='sum',
    porcentaje=lambda x: x.mean() * 100
).reset_index()

df_plan['num_dia_semana_plan'] = df_plan['fecha_plan'].dt.dayofweek
df_plan['dia_semana_plan'] = df_plan['fecha_plan'].dt.day_name()

fuera_plan_dia = df_plan.groupby(['num_dia_semana_plan', 'dia_semana_plan'])['fuera_de_plan'].agg(
    total_pedidos='count',
    pedidos_fuera='sum',
    porcentaje=lambda x: x.mean() * 100
).reset_index().sort_values('num_dia_semana_plan')

print(f"Bias Plan Global: {bias_plan_global:.4f}")
print(f"Porcentaje Fuera de Plan Total: {porcentaje_fuera_plan_total:.2f}%")
print("\nFuera de Plan por Centro de Distribución:")
print(fuera_plan_cd)
print("\nFuera de Plan por Día Planificado:")
print(fuera_plan_dia)

Bias Plan Global: 0.1062
Porcentaje Fuera de Plan Total: 17.46%

Fuera de Plan por Centro de Distribución:
  centro_distribucion  total_pedidos  pedidos_fuera  porcentaje
0           CD_CENTRO            126             14   11.111111
1             CD_ESTE            118             23   19.491525
2            CD_NORTE            123             30   24.390244
3            CD_OESTE            128             13   10.156250
4         CD_PACIFICO            122             26   21.311475
5              CD_SUR            122             23   18.852459

Fuera de Plan por Día Planificado:
   num_dia_semana_plan dia_semana_plan  total_pedidos  pedidos_fuera  \
0                    0          Monday            104             23   
1                    1         Tuesday            259             42   
2                    2       Wednesday            109             25   
3                    3        Thursday            151             21   
4                    4          Friday           

# Modelo complementario Forecast

In [15]:

lista_cds = df_reales['centro_distribucion'].unique()
registros_caracteristicas = []

for cd in lista_cds:
    df_cd = df_reales[df_reales['centro_distribucion'] == cd].sort_values('fecha_despacho').copy()
    df_cd['dia_semana'] = df_cd['fecha_despacho'].dt.dayofweek
    df_cd['es_fin_semana'] = df_cd['dia_semana'].isin([5, 6]).astype(int)
    df_cd['dia_mes'] = df_cd['fecha_despacho'].dt.day

    for retardo in [1, 2, 3, 7, 14]:
        df_cd[f'lag_{retardo}'] = df_cd['unidades_reales'].shift(retardo)

    for ventana in [7, 14, 21]:
        df_cd[f'rolling_mean_{ventana}'] = df_cd['unidades_reales'].shift(1).rolling(ventana).mean()
        df_cd[f'rolling_std_{ventana}'] = df_cd['unidades_reales'].shift(1).rolling(ventana).std()
        df_cd[f'rolling_max_{ventana}'] = df_cd['unidades_reales'].shift(1).rolling(ventana).max()

    registros_caracteristicas.append(df_cd)

df_caracteristicas = pd.concat(registros_caracteristicas, ignore_index=True)

df_dataset = pd.merge(
    df_pronostico,
    df_caracteristicas,
    on=['centro_distribucion', 'fecha_despacho'],
    how='left'
).dropna(subset=['rolling_mean_21']).sort_values('fecha_despacho').reset_index(drop=True)

columnas_features = [
    'unidades_pronosticadas', 'dia_semana', 'es_fin_semana', 'dia_mes',
    'lag_1', 'lag_2', 'lag_3', 'lag_7', 'lag_14',
    'rolling_mean_7', 'rolling_std_7', 'rolling_max_7',
    'rolling_mean_14', 'rolling_std_14', 'rolling_max_14',
    'rolling_mean_21', 'rolling_std_21', 'rolling_max_21'
]

fechas_evaluacion = [d for d in sorted(df_dataset['fecha_despacho'].unique()) if d >= pd.Timestamp('2026-02-01')]
resultados_predicciones = []

for fecha_corte in fechas_evaluacion:
    train = df_dataset[df_dataset['fecha_despacho'] < fecha_corte]
    test = df_dataset[df_dataset['fecha_despacho'] == fecha_corte]

    if len(train) < 50 or len(test) == 0:
        continue

    X_train = train[columnas_features]
    y_train = train['unidades_reales']
    X_test = test[columnas_features]

    modelo = HistGradientBoostingRegressor(random_state=42, max_iter=100)
    modelo.fit(X_train, y_train)

    test_eval = test[['centro_distribucion', 'fecha_despacho', 'horizonte', 'unidades_reales', 'unidades_pronosticadas']].copy()
    test_eval['unidades_modelo'] = modelo.predict(X_test)
    resultados_predicciones.append(test_eval)

df_evaluacion_final = pd.concat(resultados_predicciones, ignore_index=True)

wape_modelo_base = np.sum(np.abs(df_evaluacion_final['unidades_reales'] - df_evaluacion_final['unidades_pronosticadas'])) / np.sum(df_evaluacion_final['unidades_reales'])
wape_modelo_corregido = np.sum(np.abs(df_evaluacion_final['unidades_reales'] - df_evaluacion_final['unidades_modelo'])) / np.sum(df_evaluacion_final['unidades_reales'])

bias_modelo_base = np.sum(df_evaluacion_final['unidades_reales'] - df_evaluacion_final['unidades_pronosticadas']) / np.sum(df_evaluacion_final['unidades_reales'])
bias_modelo_corregido = np.sum(df_evaluacion_final['unidades_reales'] - df_evaluacion_final['unidades_modelo']) / np.sum(df_evaluacion_final['unidades_reales'])

print(f"WAPE Modelo Base: {wape_modelo_base:.4f}")
print(f"WAPE Modelo Corregido ML: {wape_modelo_corregido:.4f}")
print(f"Bias Modelo Base: {bias_modelo_base:.4f}")
print(f"Bias Modelo Corregido ML: {bias_modelo_corregido:.4f}")

WAPE Modelo Base: 0.2078
WAPE Modelo Corregido ML: 0.1077
Bias Modelo Base: 0.0662
Bias Modelo Corregido ML: 0.0125


# Regla de Resilencia (Gate de Cold Start)

In [16]:
def evaluar_resiliencia_cd(df_historico_cd, centro_nombre, umbral_dias=45):
    dias_unicos = df_historico_cd['fecha_despacho'].nunique()

    if dias_unicos < umbral_dias:
        return {
            'centro': centro_nombre,
            'estado': 'FALLBACK_BASE',
            'dias_disponibles': dias_unicos,
            'mensaje': f'Historia insuficiente ({dias_unicos}/{umbral_dias} días). Se activa modelo base.'
        }
    else:
        return {
            'centro': centro_nombre,
            'estado': 'HABILITADO_ML',
            'dias_disponibles': dias_unicos,
            'mensaje': f'Historia suficiente ({dias_unicos}/{umbral_dias} días). Se habilita modelo complementario.'
        }

evaluaciones_resiliencia = []
for cd in lista_cds:
    df_cd = df_pedidos[df_pedidos['centro_distribucion'] == cd]
    evaluaciones_resiliencia.append(evaluar_resiliencia_cd(df_cd, cd, umbral_dias=45))

df_resiliencia = pd.DataFrame(evaluaciones_resiliencia)
print(df_resiliencia)

        centro         estado  dias_disponibles  \
0    CD_CENTRO  HABILITADO_ML                60   
1      CD_ESTE  HABILITADO_ML                60   
2     CD_NORTE  HABILITADO_ML                60   
3     CD_OESTE  HABILITADO_ML                60   
4  CD_PACIFICO  HABILITADO_ML                60   
5       CD_SUR  HABILITADO_ML                60   

                                             mensaje  
0  Historia suficiente (60/45 días). Se habilita ...  
1  Historia suficiente (60/45 días). Se habilita ...  
2  Historia suficiente (60/45 días). Se habilita ...  
3  Historia suficiente (60/45 días). Se habilita ...  
4  Historia suficiente (60/45 días). Se habilita ...  
5  Historia suficiente (60/45 días). Se habilita ...  


In [19]:
class ForecastCorrectionPipeline:
    def __init__(self, umbral_dias_historia=45):
        self.umbral_dias_historia = umbral_dias_historia
        self.modelos = {}
        self.columnas_features = [
            'unidades_pronosticadas', 'dia_semana', 'es_fin_semana', 'dia_mes',
            'lag_1', 'lag_2', 'lag_3', 'lag_7', 'lag_14',
            'rolling_mean_7', 'rolling_std_7', 'rolling_max_7',
            'rolling_mean_14', 'rolling_std_14', 'rolling_max_14',
            'rolling_mean_21', 'rolling_std_21', 'rolling_max_21'
        ]

    def generar_caracteristicas(self, df_historico):
        lista_cds = df_historico['centro_distribucion'].unique()
        registros = []
        for cd in lista_cds:
            df_cd = df_historico[df_historico['centro_distribucion'] == cd].sort_values('fecha_despacho').copy()
            df_cd['dia_semana'] = df_cd['fecha_despacho'].dt.dayofweek
            df_cd['es_fin_semana'] = df_cd['dia_semana'].isin([5, 6]).astype(int)
            df_cd['dia_mes'] = df_cd['fecha_despacho'].dt.day

            for lag in [1, 2, 3, 7, 14]:
                df_cd[f'lag_{lag}'] = df_cd['unidades_reales'].shift(lag)

            for window in [7, 14, 21]:
                df_cd[f'rolling_mean_{window}'] = df_cd['unidades_reales'].shift(1).rolling(window).mean()
                df_cd[f'rolling_std_{window}'] = df_cd['unidades_reales'].shift(1).rolling(window).std()
                df_cd[f'rolling_max_{window}'] = df_cd['unidades_reales'].shift(1).rolling(window).max()

            registros.append(df_cd)
        return pd.concat(registros, ignore_index=True)

    def entrenar(self, df_pedidos, df_pronostico):
        df_pedidos['fecha_despacho'] = pd.to_datetime(df_pedidos['fecha_despacho'])
        df_pronostico['fecha_despacho'] = pd.to_datetime(df_pronostico['fecha_despacho'])

        df_reales = df_pedidos.groupby(['centro_distribucion', 'fecha_despacho'], as_index=False)['unidades_reales'].sum()
        df_caracteristicas = self.generar_caracteristicas(df_reales)

        df_full = pd.merge(
            df_pronostico,
            df_caracteristicas,
            on=['centro_distribucion', 'fecha_despacho'],
            how='left'
        ).dropna(subset=['rolling_mean_21'])

        for cd in df_full['centro_distribucion'].unique():
            df_cd = df_full[df_full['centro_distribucion'] == cd]
            dias_historia = df_pedidos[df_pedidos['centro_distribucion'] == cd]['fecha_despacho'].nunique()

            if dias_historia >= self.umbral_dias_historia:
                X = df_cd[self.columnas_features]
                y = df_cd['unidades_reales']
                modelo = HistGradientBoostingRegressor(random_state=42, max_iter=100)
                modelo.fit(X, y)
                self.modelos[cd] = modelo

    def predecir(self, df_pedidos_recientes, df_pronostico_nuevo):
        df_pedidos_recientes['fecha_despacho'] = pd.to_datetime(df_pedidos_recientes['fecha_despacho'])
        df_pronostico_nuevo['fecha_despacho'] = pd.to_datetime(df_pronostico_nuevo['fecha_despacho'])

        df_reales = df_pedidos_recientes.groupby(['centro_distribucion', 'fecha_despacho'], as_index=False)['unidades_reales'].sum()
        df_caracteristicas = self.generar_caracteristicas(df_reales)

        df_eval = pd.merge(
            df_pronostico_nuevo,
            df_caracteristicas,
            on=['centro_distribucion', 'fecha_despacho'],
            how='left'
        )

        predicciones = []
        for idx, row in df_eval.iterrows():
            cd = row['centro_distribucion']
            if cd in self.modelos and not pd.isna(row['rolling_mean_21']):
                X_single = pd.DataFrame([row[self.columnas_features]])
                pred = self.modelos[cd].predict(X_single)[0]
            else:
                pred = row['unidades_pronosticadas']
            predicciones.append(pred)

        df_pronostico_nuevo['unidades_corregidas'] = predicciones
        return df_pronostico_nuevo

# PErmutation importance

In [20]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance

df_pronostico = pd.read_csv('pronostico_demanda_generico.csv')
df_pronostico['fecha_despacho'] = pd.to_datetime(df_pronostico['fecha_despacho'])

lista_cds = df_reales['centro_distribucion'].unique()
registros_caracteristicas = []

for cd in lista_cds:
    df_cd = df_reales[df_reales['centro_distribucion'] == cd].sort_values('fecha_despacho').copy()
    df_cd['dia_semana'] = df_cd['fecha_despacho'].dt.dayofweek
    df_cd['es_fin_semana'] = df_cd['dia_semana'].isin([5, 6]).astype(int)
    df_cd['dia_mes'] = df_cd['fecha_despacho'].dt.day

    for retardo in [1, 2, 3, 7, 14]:
        df_cd[f'lag_{retardo}'] = df_cd['unidades_reales'].shift(retardo)

    for ventana in [7, 14, 21]:
        df_cd[f'rolling_mean_{ventana}'] = df_cd['unidades_reales'].shift(1).rolling(ventana).mean()
        df_cd[f'rolling_std_{ventana}'] = df_cd['unidades_reales'].shift(1).rolling(ventana).std()
        df_cd[f'rolling_max_{ventana}'] = df_cd['unidades_reales'].shift(1).rolling(ventana).max()

    registros_caracteristicas.append(df_cd)

df_caracteristicas = pd.concat(registros_caracteristicas, ignore_index=True)

df_dataset = pd.merge(
    df_pronostico,
    df_caracteristicas,
    on=['centro_distribucion', 'fecha_despacho'],
    how='left'
).dropna(subset=['rolling_mean_21']).sort_values('fecha_despacho').reset_index(drop=True)

columnas_features = [
    'unidades_pronosticadas', 'dia_semana', 'es_fin_semana', 'dia_mes',
    'lag_1', 'lag_2', 'lag_3', 'lag_7', 'lag_14',
    'rolling_mean_7', 'rolling_std_7', 'rolling_max_7',
    'rolling_mean_14', 'rolling_std_14', 'rolling_max_14',
    'rolling_mean_21', 'rolling_std_21', 'rolling_max_21'
]

X = df_dataset[columnas_features]
y = df_dataset['unidades_reales']

modelo = HistGradientBoostingRegressor(random_state=42, max_iter=100)
modelo.fit(X, y)

resultado_permutacion = permutation_importance(modelo, X, y, n_repeats=10, random_state=42)

df_importancia = pd.DataFrame({
    'caracteristica': columnas_features,
    'importancia_promedio': resultado_permutacion.importances_mean
}).sort_values('importancia_promedio', ascending=False)

print("--- Importancia de Variables en el Modelo ---")
print(df_importancia)

--- Importancia de Variables en el Modelo ---
            caracteristica  importancia_promedio
9           rolling_mean_7              0.308434
8                   lag_14              0.134303
0   unidades_pronosticadas              0.082142
7                    lag_7              0.057280
6                    lag_3              0.022328
1               dia_semana              0.004119
4                    lag_1              0.003245
12         rolling_mean_14              0.003110
10           rolling_std_7              0.002791
15         rolling_mean_21              0.002643
16          rolling_std_21              0.002557
3                  dia_mes              0.002154
5                    lag_2              0.001247
13          rolling_std_14              0.001150
17          rolling_max_21              0.000198
11           rolling_max_7              0.000185
14          rolling_max_14              0.000052
2            es_fin_semana              0.000000


# Shap

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor

# 1. Asegurar la instalación de shap si no la tienes instalada
try:
    import shap
except ImportError:
    !pip install shap
    import shap

# 2. Cargar datasets y preparar la matriz de características
df_pedidos = pd.read_csv('pedidos_historicos_genericos.csv')
df_pronostico = pd.read_csv('pronostico_demanda_generico.csv')

df_pedidos['fecha_despacho'] = pd.to_datetime(df_pedidos['fecha_despacho'])
df_pronostico['fecha_despacho'] = pd.to_datetime(df_pronostico['fecha_despacho'])

df_reales = df_pedidos.groupby(['centro_distribucion', 'fecha_despacho'], as_index=False)['unidades_reales'].sum()

lista_cds = df_reales['centro_distribucion'].unique()
registros_caracteristicas = []

for cd in lista_cds:
    df_cd = df_reales[df_reales['centro_distribucion'] == cd].sort_values('fecha_despacho').copy()
    df_cd['dia_semana'] = df_cd['fecha_despacho'].dt.dayofweek
    df_cd['es_fin_semana'] = df_cd['dia_semana'].isin([5, 6]).astype(int)
    df_cd['dia_mes'] = df_cd['fecha_despacho'].dt.day

    for retardo in [1, 2, 3, 7, 14]:
        df_cd[f'lag_{retardo}'] = df_cd['unidades_reales'].shift(retardo)

    for ventana in [7, 14, 21]:
        df_cd[f'rolling_mean_{ventana}'] = df_cd['unidades_reales'].shift(1).rolling(ventana).mean()
        df_cd[f'rolling_std_{ventana}'] = df_cd['unidades_reales'].shift(1).rolling(ventana).std()
        df_cd[f'rolling_max_{ventana}'] = df_cd['unidades_reales'].shift(1).rolling(ventana).max()

    registros_caracteristicas.append(df_cd)

df_caracteristicas = pd.concat(registros_caracteristicas, ignore_index=True)

df_dataset = pd.merge(
    df_pronostico,
    df_caracteristicas,
    on=['centro_distribucion', 'fecha_despacho'],
    how='left'
).dropna(subset=['rolling_mean_21']).sort_values('fecha_despacho').reset_index(drop=True)

columnas_features = [
    'unidades_pronosticadas', 'dia_semana', 'es_fin_semana', 'dia_mes',
    'lag_1', 'lag_2', 'lag_3', 'lag_7', 'lag_14',
    'rolling_mean_7', 'rolling_std_7', 'rolling_max_7',
    'rolling_mean_14', 'rolling_std_14', 'rolling_max_14',
    'rolling_mean_21', 'rolling_std_21', 'rolling_max_21'
]

X = df_dataset[columnas_features]
y = df_dataset['unidades_reales']

# 3. Entrenar el modelo
modelo = HistGradientBoostingRegressor(random_state=42, max_iter=100)
modelo.fit(X, y)

# 4. Explicabilidad con SHAP
# Usamos TreeExplainer o PermutationExplainer/Explainer según la compatibilidad del estimador
explainer = shap.Explainer(modelo.predict, X)
shap_values = explainer(X)

# 5. Generar y mostrar el gráfico SHAP Beeswarm
plt.figure(figsize=(10, 6))
shap.plots.beeswarm(shap_values, max_display=12, show=True)

PermutationExplainer explainer:  40%|███▉      | 373/936 [04:24<06:22,  1.47it/s]